In [ ]:
# Load necessary libraries
from pathlib import Path

import moftransformer

In [5]:
root_dataset = 'root_dataset/'
downstream = 'D'

max_epochs = 20
batch_size = 32
mean = -5
std = 1

lr_range = [0.5, 1, 2, 5, 10]
wd_range = [0.2, 0.5, 1, 2, 5]

ref_lr = 1e-4
ref_wd = 1e-2

In [ ]:
results_dict = {}

for lr_scale in lr_range:
    for wd_scale in wd_range:
        lr = ref_lr * lr_scale
        wd = ref_wd * wd_scale

        log_dir = f'logs_lr_{lr}_wd_{wd}/'

        moftransformer.run(root_dataset, downstream, log_dir=log_dir,
                        load_path=None, max_epochs=max_epochs, batch_size=batch_size,
                        mean=mean, std=std, learning_rate=lr, weight_decay=wd)
        
        # get ckpt file and gather prediction data
        seed=0
        version=0
        checkpoint='best'

        load_path = Path(log_dir) / f'pretrained_mof_seed{seed}_from_/version_{version}/checkpoints/{checkpoint}.ckpt'

        if not load_path.exists():
            raise ValueError(f'load_path does not exists. check path for .ckpt file : {load_path}')
            
        moftransformer.predict(
            root_dataset, load_path=load_path, downstream=downstream, split='train', mean=mean, std=std
        )
        moftransformer.predict(
            root_dataset, load_path=load_path, downstream=downstream, split='val', mean=mean, std=std
        )